# Notebook 13: Deep Deterministic Policy Gradient (DDPG)

## Motivation

So far, we've learned:
- **DQN (Notebook 10)**: Off-policy learning for discrete actions using Q-learning
- **PPO (Notebook 11)**: On-policy actor-critic for continuous actions with clipped surrogate loss
- **SAC (Notebook 12)**: Off-policy actor-critic with maximum entropy and automatic temperature tuning

**DDPG** (*Deep Deterministic Policy Gradient*) bridges DQN and actor-critic:
- Like DQN: off-policy learning with experience replay and target networks
- Like A2C: separate actor and critic networks
- **Novel**: deterministic actor (outputs a single action, not a distribution)
- **Exploration**: Ornstein-Uhlenbeck noise instead of entropy

This notebook teaches DDPG from first principles, then implements it step-by-step.

## Setup

In [ ]:
import sys
sys.path.append('.')

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

from src.environments import InvertedPendulumEnv
from src.policies import DDPGPolicy, OUNoise

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Part 1: Core Concepts

### Deterministic vs. Stochastic Policies

We've seen two types of policies:

| Policy Type | Example | Output | Exploration |
|---|---|---|---|
| **Stochastic** | PPO, SAC, A2C | π(a\|s) distribution | Built-in (entropy, noise) |
| **Deterministic** | DDPG, TD3 | μ(s) single action | External (OU noise, ε-greedy) |

**DDPG uses a deterministic policy**: instead of sampling from π(a\|s),
we directly output a = μ_θ(s).

**Advantage**: The actor is simpler (no need to sample or compute log-probabilities).

**Trade-off**: Exploration must be handled separately (OU noise during training).

### Deterministic Policy Gradient Theorem

The key insight is the **Deterministic Policy Gradient (DPG) Theorem**:

Instead of:
$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_a \log \pi(a|s) \cdot Q(s, a)]$$

We have (for deterministic μ_θ):
$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_a Q(s, μ(s)) \cdot \nabla_\theta μ_\theta(s)]$$

**Why is this simpler?**
- No need to compute log π(a\|s) (which requires sampling)
- Gradients flow directly from the Q-value through the actor
- More sample-efficient than stochastic policies

**Trade-off**:
- Only off-policy (can't easily do on-policy with deterministic policies)
- Needs explicit exploration (OU noise)

### DDPG Algorithm Overview

```
Initialize:
  - Actor network μ_θ(s) with parameters θ
  - Critic network Q_φ(s, a) with parameters φ
  - Target actor μ'_θ(s) and critic Q'_φ(s, a) (copies of above)
  - Experience replay buffer B
  - OU noise process

Loop for each episode:
  1. Initialize state s
  2. For each step t:
     a) Sample action with noise: a = μ(s) + OU_noise
     b) Execute action, observe r, s'
     c) Store (s, a, r, s') in replay buffer
     d) Sample random mini-batch from B
     e) Compute TD target: y = r + γ Q'(s', μ'(s'))
     f) Update critic: φ ← φ - α_c ∇(y - Q(s, a))²
     g) Update actor: θ ← θ + α_a ∇ Q(s, μ(s))
     h) Soft update targets: θ' ← τ θ + (1-τ) θ', φ' ← τ φ + (1-τ) φ'
```

### Ornstein-Uhlenbeck Noise

DDPG uses OU noise for exploration instead of random Gaussian noise.

**Why OU noise?**
- Gaussian noise is uncorrelated → jerky, unrealistic actions
- OU noise is temporally correlated → smooth, natural trajectories
- Inspired by physics: dθ = θ(μ - x)dt + σ dW

**Properties**:
- Reverts to mean μ over time (drift term)
- Has smooth autocorrelation (not white noise)
- Better for physical control problems

In [ ]:
# Demo: OU noise vs Gaussian noise
ou = OUNoise(action_dim=1, theta=0.15, sigma=0.3, dt=0.01)
gaussian = np.random.randn(200)
ou_samples = [ou.sample()[0] for _ in range(200)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gaussian, label='Gaussian noise', alpha=0.7)
axes[0].set_title('Gaussian Noise (Uncorrelated)')
axes[0].set_xlabel('Time step')
axes[0].set_ylabel('Noise value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ou_samples, label='OU noise', alpha=0.7, color='orange')
axes[1].set_title('Ornstein-Uhlenbeck Noise (Correlated)')
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Noise value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Gaussian autocorr at lag 1: {np.corrcoef(gaussian[:-1], gaussian[1:])[0, 1]:.3f}')
print(f'OU autocorr at lag 1:       {np.corrcoef(ou_samples[:-1], ou_samples[1:])[0, 1]:.3f}')

## Part 2: Implementation Deep Dive

### Actor Network (Deterministic Policy)

The actor is a simple feedforward network:
```
s → Dense(64) → ReLU → Dense(64) → ReLU → Dense(1) → Tanh → a
```

Key points:
- **Tanh output**: maps to [-1, 1], then scaled to [action_low, action_high]
- **No stochasticity**: same input → same output (deterministic)
- **Very simple**: no log-probabilities, no sampling

In [ ]:
# Visualize determinism of actor
from src.policies.ddpg_policy import DDPGActorNet

actor = DDPGActorNet(state_dim=4, action_dim=1, hidden_sizes=[64, 64])
actor.eval()

state = torch.tensor([0.0, 0.0, 0.1, 0.0]).unsqueeze(0)

with torch.no_grad():
    actions = [actor(state).item() for _ in range(5)]

print(f'Action outputs for same input: {actions}')
print(f'All identical? {all(a == actions[0] for a in actions)}')

### Critic Network (Q-function)

The critic estimates Q(s, a):
```
[s, a] → Concat → Dense(64) → ReLU → Dense(64) → ReLU → Dense(1) → Q(s, a)
```

Key points:
- **Concatenates state and action**: unlike Q-networks in DQN, here action is an input
- **Output is Q-value**: unbounded (no activation on final layer)
- **Gradients flow through both s and a**: allows computing ∇_a Q for the actor update

In [ ]:
# Visualize actor-critic connection
from src.policies.ddpg_policy import DDPGCriticNet

critic = DDPGCriticNet(state_dim=4, action_dim=1, hidden_sizes=[64, 64])

# Example: Q-values increase with 'better' actions
state = torch.tensor([[0.0, 0.0, 0.1, 0.0]] * 5)
actions = torch.tensor([[-1.0], [-0.5], [0.0], [0.5], [1.0]])

with torch.no_grad():
    q_values = critic(state, actions).squeeze()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([-1.0, -0.5, 0.0, 0.5, 1.0], q_values.numpy(), 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Action', fontsize=12)
ax.set_ylabel('Q(s, a)', fontsize=12)
ax.set_title('Critic Network: Q-value vs Action', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Target Networks and Soft Updates

Like DQN, DDPG uses target networks to stabilize learning.
Unlike DQN (hard updates every N steps), DDPG uses **soft (Polyak) updates** every step:

$$\theta^{-} \leftarrow \tau \theta + (1-\tau) \theta^{-}$$

**Advantages**:
- Smoother target updates (no sudden jumps)
- No need to decide when to update
- Better stability (proven empirically)

With τ = 0.005: target networks lag far behind main networks,
giving stable TD targets while allowing slow, smooth adaptation.

## Part 3: Training on InvertedPendulum

Now let's train DDPG and compare with previous algorithms.

In [ ]:
# Initialize environment and policy
env = InvertedPendulumEnv()
policy = DDPGPolicy(
    hidden_sizes=[64, 64],
    action_low=-10.0,
    action_high=10.0,
    gamma=0.99,
    tau=0.005,
    actor_lr=1e-4,
    critic_lr=1e-3,
    warmup_steps=1000,  # Random exploration phase
)

print(f'Actor parameters: {sum(p.numel() for p in policy.actor.parameters()):,}')
print(f'Critic parameters: {sum(p.numel() for p in policy.critic.parameters()):,}')
print(f'Warmup steps: {policy.warmup_steps}')
print(f'Replay buffer capacity: {policy.replay_buffer.capacity:,}')

In [ ]:
# Training loop
n_episodes = 200
episode_rewards = []
actor_losses = []
critic_losses = []
avg_rewards = []

pbar = tqdm(range(n_episodes), desc='Training DDPG')

for episode in pbar:
    state, _ = env.reset()
    episode_reward = 0.0
    
    for step in range(500):  # Max steps per episode
        # Select action
        action = policy.get_action_train(state)
        state_next, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Store transition
        policy.push(state, action, reward, state_next, done)
        
        # Train
        if policy.step_count > policy.warmup_steps and len(policy.replay_buffer) > 64:
            stats = policy.update(batch_size=64, n_updates=1)
            actor_losses.append(stats['actor_loss'])
            critic_losses.append(stats['critic_loss'])
        
        episode_reward += reward
        state = state_next
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    if len(episode_rewards) >= 10:
        avg_reward = np.mean(episode_rewards[-10:])
        avg_rewards.append(avg_reward)
    
    pbar.set_postfix({'reward': episode_reward:.1f}, refresh=True)

print(f'\nTraining complete!')
print(f'Final 10-episode average reward: {np.mean(episode_rewards[-10:]):.1f}')

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward curve
axes[0].plot(episode_rewards, alpha=0.5, label='Episode reward')
axes[0].plot(avg_rewards, linewidth=2, label='10-ep avg')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('DDPG: Reward Learning Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Critic loss
if critic_losses:
    axes[1].plot(critic_losses, alpha=0.7)
    axes[1].set_xlabel('Update step')
    axes[1].set_ylabel('Critic Loss (MSE)')
    axes[1].set_title('DDPG: Critic Loss')
    axes[1].grid(True, alpha=0.3)

# Actor loss
if actor_losses:
    axes[2].plot(actor_losses, alpha=0.7, color='orange')
    axes[2].set_xlabel('Update step')
    axes[2].set_ylabel('Actor Loss (negative Q)')
    axes[2].set_title('DDPG: Actor Loss')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4: Comparison with Other Algorithms

Let's compare DDPG with PPO and SAC on the same task.

| Algorithm | On/Off-Policy | Actor Deterministic | Exploration | Stability |
|-----------|---|---|---|---|
| **PPO (11)** | On | No | Entropy | Good |
| **SAC (12)** | Off | No | Entropy | Very good |
| **DDPG (13)** | Off | Yes | OU noise | Good |

**DDPG Strengths**:
- Off-policy → sample efficient
- Deterministic actor → simpler, faster inference
- Soft target updates → smooth training

**DDPG Weaknesses**:
- Deterministic policy harder to explore with
- Can be unstable in high-dimensional state spaces
- Sensitive to hyperparameters (learning rates, noise scales)

## Part 5: Hands-On Experiments

### Experiment 1: Effect of OU Noise Scale

In [ ]:
# Try different noise scales
noise_scales = [0.1, 0.3, 0.5]
results = {}

for sigma in noise_scales:
    policy = DDPGPolicy(
        warmup_steps=500,
        actor_lr=1e-4,
        critic_lr=1e-3,
    )
    policy.ou_noise = OUNoise(action_dim=1, sigma=sigma)
    
    env = InvertedPendulumEnv()
    episode_rewards = []
    
    for episode in tqdm(range(100), desc=f'σ={sigma}', leave=False):
        state, _ = env.reset()
        episode_reward = 0.0
        
        for step in range(500):
            action = policy.get_action_train(state)
            state_next, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            policy.push(state, action, reward, state_next, done)
            
            if policy.step_count > 500 and len(policy.replay_buffer) > 64:
                policy.update(batch_size=64, n_updates=1)
            
            episode_reward += reward
            state = state_next
            if done:
                break
        
        episode_rewards.append(episode_reward)
    
    results[sigma] = episode_rewards

# Plot results
plt.figure(figsize=(10, 5))
for sigma, rewards in results.items():
    avg = [np.mean(rewards[max(0, i-10):i+1]) for i in range(len(rewards))]
    plt.plot(avg, label=f'σ={sigma}', linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Average Reward (10-episode)')
plt.title('DDPG: Effect of OU Noise Scale')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Effect of noise scale:')
for sigma, rewards in results.items():
    print(f'  σ={sigma}: final avg = {np.mean(rewards[-10:]):.1f}')

### Experiment 2: Effect of Target Update Rate (τ)

In [ ]:
# Try different tau values
tau_values = [0.001, 0.005, 0.01]
results = {}

for tau in tau_values:
    policy = DDPGPolicy(
        warmup_steps=500,
        tau=tau,
        actor_lr=1e-4,
        critic_lr=1e-3,
    )
    
    env = InvertedPendulumEnv()
    episode_rewards = []
    
    for episode in tqdm(range(100), desc=f'τ={tau}', leave=False):
        state, _ = env.reset()
        episode_reward = 0.0
        
        for step in range(500):
            action = policy.get_action_train(state)
            state_next, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            policy.push(state, action, reward, state_next, done)
            
            if policy.step_count > 500 and len(policy.replay_buffer) > 64:
                policy.update(batch_size=64, n_updates=1)
            
            episode_reward += reward
            state = state_next
            if done:
                break
        
        episode_rewards.append(episode_reward)
    
    results[tau] = episode_rewards

# Plot results
plt.figure(figsize=(10, 5))
for tau, rewards in results.items():
    avg = [np.mean(rewards[max(0, i-10):i+1]) for i in range(len(rewards))]
    plt.plot(avg, label=f'τ={tau}', linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Average Reward (10-episode)')
plt.title('DDPG: Effect of Target Update Rate (τ)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Effect of target update rate:')
for tau, rewards in results.items():
    print(f'  τ={tau}: final avg = {np.mean(rewards[-10:]):.1f}')

## Part 6: Practice Questions

1. **Deterministic Policy Gradient**: Why can we compute ∇_θ J(θ) = E[∇_a Q(s, μ(s)) · ∇_θ μ(s)]?
   - Hint: Compare this to REINFORCE (∇_θ log π · Q) and Actor-Critic (∇_θ log π · A).

2. **OU Noise**: Why is OU noise better than Gaussian noise for continuous control?
   - Implement an experiment comparing the two over 100 episodes.

3. **Soft Target Updates**: What happens if we set τ = 1.0? τ = 0.0?
   - τ = 1: Target network is updated immediately (equivalent to hard update every step)
   - τ = 0: Target network never updates (stays at initialization)

4. **Off-Policy Learning**: Why must DDPG be off-policy (use a replay buffer)?
   - Hint: Think about what happens with a deterministic actor and on-policy data.

5. **Comparison**: When would you choose DDPG over PPO or SAC?
   - DDPG: When you want a simple deterministic policy and have continuous actions
   - PPO: When you want on-policy learning and strong exploration
   - SAC: When you want maximum entropy and automatic temperature tuning

## Summary

**DDPG (Deep Deterministic Policy Gradient)**:
- Combines benefits of DQN (off-policy replay buffer) and Actor-Critic (continuous actions)
- Uses a **deterministic actor** μ_θ(s) that outputs a single action
- Exploration via **Ornstein-Uhlenbeck noise** (temporally correlated)
- **Soft target updates** via Polyak averaging for stability
- More sample-efficient than on-policy methods
- Simpler actor network (no sampling, no log-probabilities)

**Key innovations**:
1. Deterministic Policy Gradient Theorem
2. Experience replay for continuous actions
3. Soft (Polyak) target updates
4. OU noise for realistic exploration

**Next steps**:
- Implement TD3 (Twin Delayed DDPG) for better stability
- Try on more complex environments (MuJoCo)
- Study SAC's maximum entropy framework